# SYMFLUENCE Tutorial 04a — Logan River Workshop (Lumped, Cloud Data)

## Introduction

This workshop notebook demonstrates basin-scale hydrological modeling using SYMFLUENCE's lumped representation approach for the Logan River at Logan, Utah using cloud-based data sources. The workflow includes:

1. **Configuration** — Set up a lumped basin model for the Logan River
2. **Domain Definition** — Delineate the watershed using TauDEM
3. **Data Acquisition** — Fetch AORC forcing data and USGS streamflow observations from cloud sources
4. **Model Execution** — Run a hydrological model (SUMMA, HBV, SAC-SMA, Xinanjiang, HEC-HMS, or TOPMODEL)
5. **Evaluation & Calibration** — Assess model performance and calibrate parameters

The **Logan River at Logan** is a snow-dominated mountain watershed in the Bear River Range of the Wasatch Mountains. USGS station 10109000 provides streamflow observations. The watershed covers approximately 218 km² with elevations ranging from ~1,400 m to over 2,900 m.

### Switching Models

To use a different model, change the `model=` parameter in the configuration cell below. Supported models include: `'SUMMA'`, `'HBV'`, `'SACSMA'`, `'XINANJIANG'`, `'HECHMS'`, `'TOPMODEL'`.

### 2i2c Environment Setup

This notebook is designed to work with the 2i2c JupyterHub environment using the pre-installed SYMFLUENCE virtual environment at `/tmp/symfluence`.

**Launch this notebook from the CLI:**
```bash
symfluence example launch 4a
```

In [ ]:
# Environment verification
import sys
import warnings
from pathlib import Path

# Suppress experimental module warnings for cleaner output
warnings.filterwarnings('ignore', message='.*is an EXPERIMENTAL module.*')
warnings.filterwarnings('ignore', message='.*import failed.*')

print(f"Python executable: {sys.executable}")

# Verify SYMFLUENCE is available
try:
    import symfluence
    print(f"SYMFLUENCE version: {symfluence.__version__}")
    print(f"SYMFLUENCE location: {Path(symfluence.__file__).parent}")
except ImportError:
    print("ERROR: SYMFLUENCE not found. Please activate the symfluence environment.")
    sys.exit(1)

In [ ]:
# Fix working directory if running from .ipynb_checkpoints
import os
from pathlib import Path

current_dir = Path.cwd()
print(f"Current directory: {current_dir}")

# If we're in .ipynb_checkpoints, move up to parent directory
if '.ipynb_checkpoints' in str(current_dir):
    correct_dir = current_dir.parent
    os.chdir(correct_dir)
    print(f"Changed to: {Path.cwd()}")
else:
    print("Working directory is correct")

# Verify we're in the workshop notebooks directory
expected_notebook = Path.cwd() / '04a_logan_river_workshop.ipynb'
if not expected_notebook.exists():
    print(f"WARNING: Expected notebook not found at {expected_notebook}")
else:
    print("Notebook location verified")

## Step 1 — Configuration

Create a configuration for the Logan River lumped basin model. Key settings:
- **Domain**: Lumped (single HRU) representation
- **Forcing**: AORC (Analysis of Record for Calibration) — 1km hourly gridded data
- **Observations**: USGS streamflow from station 10109000
- **Period**: 4 years (2018-2021) with 1-year spinup

In [ ]:
# Step 1 — Create basin-scale configuration using new config system

from pathlib import Path
from symfluence import SYMFLUENCE
from symfluence.core.config.models import SymfluenceConfig
import os

# === Logan River Basin Configuration ===

# Ensure we're working from the correct directory
current_dir = Path.cwd()
print(f"INITIAL working directory: {current_dir}")

if '.ipynb_checkpoints' in str(current_dir):
    current_dir = current_dir.parent
    os.chdir(current_dir)
    print(f"CHANGED to: {Path.cwd()}")
else:
    print("Working directory is correct")

# Derive paths dynamically from repo structure
# Notebooks live at: <repo>/examples/04_workshop_notebooks/
repo_root = current_dir.parent.parent
data_dir = repo_root.parent / 'SYMFLUENCE_data'

print(f"Notebook directory: {current_dir}")
print(f"Repo root (CODE_DIR): {repo_root}")
print(f"Data directory (DATA_DIR): {data_dir}")

# ============================================================================
# MODEL SELECTION — Change this to switch models
# Supported: 'SUMMA', 'HBV', 'SACSMA', 'XINANJIANG', 'HECHMS', 'TOPMODEL'
# ============================================================================
SELECTED_MODEL = 'SUMMA'

# Models that need a routing model
SUMMA_MODELS = {'SUMMA'}
routing_model = 'mizuRoute' if SELECTED_MODEL in SUMMA_MODELS else None

# Build config kwargs
config_kwargs = dict(
    # Basic identification
    domain_name='Logan_River_at_Logan',
    experiment_id='workshop_run_1',

    # Paths
    SYMFLUENCE_DATA_DIR=str(data_dir),
    SYMFLUENCE_CODE_DIR=str(repo_root),
    TAUDEM_DIR='default',

    # Model
    model=SELECTED_MODEL,

    # Simulation period (4 years: 2018-2021)
    time_start='2018-01-01 01:00',
    time_end='2021-12-31 23:00',
    spinup_period='2018-01-01, 2018-12-31',
    calibration_period='2019-01-01, 2020-12-31',
    evaluation_period='2021-01-01, 2021-12-31',

    # Spatial domain (Logan River at Logan, UT — USGS 10109000)
    pour_point_coords='41.743098/-111.786432',
    bounding_box_coords='42.15/-111.90/41.70/-111.40',
    definition_method='lumped',
    discretization='GRUs',
    lumped_watershed_method='TauDEM',

    # Data sources
    data_access='cloud',
    forcing_dataset='AORC',
    forcing_measurement_height=10,
    dem_source='copernicus',
    download_dem=True,

    # Streamflow observations
    station_id='10109000',
    streamflow_data_provider='USGS',
    download_usgs_data=True,

    # Calibration (applies to all models)
    OPTIMIZATION_METHODS=['iteration'],
    optimization_target='streamflow',
    ITERATIVE_OPTIMIZATION_ALGORITHM='DDS',
    optimization_metric='KGE',
    calibration_timestep='hourly',
    iterations=100,
)

# Add routing model if needed
if routing_model:
    config_kwargs['routing_model'] = routing_model
    config_kwargs['SUMMA_INSTALL_PATH'] = 'default'
    config_kwargs['MIZUROUTE_INSTALL_PATH'] = 'default'

# Add SUMMA-specific calibration params (JAX models use their own parameter managers)
if SELECTED_MODEL == 'SUMMA':
    config_kwargs['params_to_calibrate'] = 'k_soil,theta_sat,aquiferBaseflowExp,aquiferBaseflowRate,qSurfScale,summerLAI,frozenPrecipMultip,Fcapil,tempCritRain,heightCanopyTop,heightCanopyBottom,windReductionParam,vGn_n'
    config_kwargs['basin_params_to_calibrate'] = 'routingGammaScale,routingGammaShape'

config = SymfluenceConfig.from_minimal(**config_kwargs)

print(f"\nConfiguration:")
print(f"  Model: {SELECTED_MODEL}")
print(f"  Routing: {routing_model or 'none'}")

# Save configuration
config_path = Path('./config_logan_river_lumped.yaml')
config_dict = config.to_dict(flatten=True)
import yaml
with open(config_path, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)
print(f"  Config saved to: {config_path}")

# Initialize SYMFLUENCE
symfluence = SYMFLUENCE(config)
project_dir = symfluence.managers['project'].setup_project()
pour_point_path = symfluence.managers['project'].create_pour_point()

print(f"\nProject structure created at: {project_dir}")
print(f"Pour point shapefile: {pour_point_path}")
print("=" * 80)

## Step 2 — Domain Definition

Delineate the Logan River watershed using TauDEM and create a single lumped HRU.

### Step 2a — Geospatial Attribute Acquisition

Acquire elevation, land cover, and soil data from cloud sources.

In [ ]:
# Step 2a — Acquire geospatial attributes from cloud
symfluence.managers['data'].acquire_attributes()
print("Attribute acquisition complete")

### Step 2b — Watershed Delineation

Delineate the watershed boundary from the pour point using TauDEM.

In [ ]:
# Step 2b — Watershed delineation
watershed_path = symfluence.managers['domain'].define_domain()
print(f"Watershed delineation complete")
print(f"Watershed file: {watershed_path}")

### Step 2c — Domain Discretization

Create a single lumped HRU for the watershed.

In [ ]:
# Step 2c — Discretization (single lumped HRU)
hru_path = symfluence.managers['domain'].discretize_domain()
print("Domain discretization complete")
print(f"HRU file: {hru_path}")

### Step 2d — Visualization

Visualize the delineated watershed and pour point.

In [ ]:
# Step 2d — Basin visualization

import geopandas as gpd
import matplotlib.pyplot as plt

# Load spatial data
basin_path = project_dir / 'shapefiles' / 'river_basins' / f"{config.domain.name}_riverBasins_lumped.shp"
hru_file = project_dir / 'shapefiles' / 'catchment' / 'lumped' / config.domain.experiment_id / f"{config.domain.name}_HRUs_GRUs.shp"                     

watershed_gdf = gpd.read_file(str(basin_path))
hru_gdf = gpd.read_file(str(hru_file))
pour_point_gdf = gpd.read_file(pour_point_path)

# Calculate area (UTM Zone 12N for Utah)
watershed_proj = watershed_gdf.to_crs('EPSG:32612')
area_km2 = watershed_proj.geometry.area.sum() / 1e6

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
watershed_gdf.boundary.plot(ax=ax, color='blue', linewidth=2, label='Watershed')
hru_gdf.plot(ax=ax, facecolor='lightblue', edgecolor='blue', alpha=0.3)
pour_point_gdf.plot(ax=ax, color='red', markersize=150, marker='*', label=f'Pour Point (USGS {config.evaluation.streamflow.station_id})')

ax.set_title(f"Logan River at Logan\nArea: {area_km2:.0f} km²", fontweight='bold', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

print(f"Watershed area: {area_km2:.0f} km²")
print(f"Number of HRUs: {len(hru_gdf)} (lumped)")

## Step 3 — Data Acquisition and Preprocessing

Fetch forcing data (AORC) and streamflow observations (USGS) from cloud sources.

### Step 3a — USGS Streamflow Observations

Download and process USGS streamflow data for station 10109000.

In [ ]:
# Step 3a — Download and process USGS streamflow data
symfluence.managers['data'].process_observed_data()                                                                                                      

print("USGS streamflow data acquisition complete")

### Step 3b — AORC Meteorological Forcing

Download AORC forcing data from NOAA's cloud archive (AWS S3). AORC provides:
- 1 km spatial resolution
- Hourly temporal resolution
- Complete forcing variables: precipitation, temperature, humidity, wind, radiation, pressure

In [ ]:
# Step 3b — Acquire AORC forcing data from cloud
symfluence.managers['data'].acquire_forcings()
print("AORC forcing acquisition complete")

### Step 3c — Model-Agnostic Preprocessing

Standardize forcing data: variable names, units, and spatial averaging over the watershed.

In [ ]:
# Step 3c — Model-agnostic preprocessing
symfluence.managers['data'].run_model_agnostic_preprocessing()
print("Model-agnostic preprocessing complete")

## Step 4 — Model Configuration and Execution

Run model-specific preprocessing and execute the selected model.

In [ ]:
# Step 4a — Model-specific preprocessing
print(f"Preprocessing for {config.model.hydrological_model}...")
symfluence.managers['model'].preprocess_models()
print(f"{config.model.hydrological_model} preprocessing complete")

In [ ]:
# Step 4b — Model execution
model_name = config.model.hydrological_model
routing_name = config.model.routing_model or 'no routing'
print(f"Running {model_name} with {routing_name}...")
print(f"Simulation period: {config.domain.time_start} to {config.domain.time_end}")
symfluence.managers['model'].run_models()
print("Model execution complete")

# Step 4c — Post-processing (extract streamflow to standardized results CSV)
symfluence.managers['model'].postprocess_results()
print("Post-processing complete")

## Step 5 — Streamflow Evaluation

Compare simulated streamflow against USGS observations using standard hydrological metrics.

In [ ]:
# Step 5 — Streamflow evaluation (model-agnostic)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

model_name = config.model.hydrological_model
experiment_id = config.domain.experiment_id

# Load results CSV (written by postprocessor)
results_file = project_dir / "results" / f"{experiment_id}_results.csv"
if not results_file.exists():
    raise FileNotFoundError(f"No results file found at: {results_file}")

results_df = pd.read_csv(results_file, index_col=0, parse_dates=True)
print(f"Loaded results: {results_file}")
print(f"Columns: {list(results_df.columns)}")

# Find simulation column (any *_discharge_cms column)
sim_cols = [c for c in results_df.columns if 'discharge' in c.lower() and 'obs' not in c.lower()]
if not sim_cols:
    raise ValueError(f"No discharge column found. Available: {list(results_df.columns)}")
sim_col = sim_cols[0]
print(f"Using simulation column: {sim_col}")

# Load observed streamflow
obs_dir = project_dir / "data" / "observations" / "streamflow" / "preprocessed"
obs_path = obs_dir / f"{config.domain.name}_streamflow_processed.csv"
if obs_path.exists():
    obs_df = pd.read_csv(obs_path)
    obs_df['datetime'] = pd.to_datetime(obs_df['datetime'], dayfirst=True, utc=True, errors='coerce').dt.tz_convert(None)
    obs_df.set_index('datetime', inplace=True)
    obs_daily = obs_df['discharge_cms'].resample('D').mean()
else:
    print(f"Warning: observations not found at {obs_path}")
    obs_daily = None

# Align simulation and observations
sim_series = results_df[sim_col].resample('D').mean()

# Exclude spinup period
spinup_end = pd.to_datetime(config.domain.spinup_period.split(',')[1].strip())
sim_series = sim_series[sim_series.index > spinup_end]

if obs_daily is not None:
    common_idx = sim_series.index.intersection(obs_daily.index)
    obs_valid = obs_daily.loc[common_idx].dropna()
    sim_valid = sim_series.loc[obs_valid.index]

    # Remove NaN pairs
    mask = ~(np.isnan(obs_valid.values) | np.isnan(sim_valid.values))
    obs_clean = obs_valid[mask]
    sim_clean = sim_valid[mask]

    print(f"\nExcluding spinup up to: {spinup_end}")
    print(f"Evaluation period: {obs_clean.index[0]} to {obs_clean.index[-1]}")
    print(f"Valid data points: {len(obs_clean)}")

    # Metrics
    def nse(obs, sim):
        return float(1 - np.sum((obs - sim)**2) / np.sum((obs - obs.mean())**2))

    def kge(obs, sim):
        r = obs.corr(sim)
        alpha = sim.std() / obs.std()
        beta = sim.mean() / obs.mean()
        return float(1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2))

    def pbias(obs, sim):
        return float(100 * (sim.sum() - obs.sum()) / obs.sum())

    metrics = {
        'NSE': round(nse(obs_clean, sim_clean), 3),
        'KGE': round(kge(obs_clean, sim_clean), 3),
        'PBIAS': round(pbias(obs_clean, sim_clean), 1)
    }

    print(f"\nPerformance Metrics ({model_name}, Uncalibrated):")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
else:
    print("No observations available for evaluation")
    obs_clean = sim_clean = metrics = None

In [ ]:
# Visualization
if obs_clean is not None and metrics is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Time series (top left)
    axes[0, 0].plot(obs_clean.index, obs_clean.values, 'b-', label='Observed (USGS)', linewidth=1.2, alpha=0.7)
    axes[0, 0].plot(sim_clean.index, sim_clean.values, 'r-', label=f'Simulated ({model_name})', linewidth=1.2, alpha=0.7)
    axes[0, 0].set_ylabel('Discharge (m\u00b3/s)')
    axes[0, 0].set_title('Streamflow Time Series')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].text(0.02, 0.95, f"NSE: {metrics['NSE']}\nKGE: {metrics['KGE']}\nBias: {metrics['PBIAS']}%",
                    transform=axes[0, 0].transAxes, verticalalignment='top',
                    bbox=dict(facecolor='white', alpha=0.8), fontsize=9)

    # Scatter (top right)
    axes[0, 1].scatter(obs_clean, sim_clean, alpha=0.5, s=10)
    max_val = max(obs_clean.max(), sim_clean.max())
    axes[0, 1].plot([0, max_val], [0, max_val], 'k--', alpha=0.5)
    axes[0, 1].set_xlabel('Observed (m\u00b3/s)')
    axes[0, 1].set_ylabel('Simulated (m\u00b3/s)')
    axes[0, 1].set_title('Observed vs Simulated')
    axes[0, 1].grid(True, alpha=0.3)

    # Monthly climatology (bottom left)
    monthly_obs = obs_clean.groupby(obs_clean.index.month).mean()
    monthly_sim = sim_clean.groupby(sim_clean.index.month).mean()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[1, 0].plot(monthly_obs.index, monthly_obs.values, 'b-o', label='Observed', markersize=6)
    axes[1, 0].plot(monthly_sim.index, monthly_sim.values, 'r-o', label='Simulated', markersize=6)
    axes[1, 0].set_xticks(range(1, 13))
    axes[1, 0].set_xticklabels(month_names)
    axes[1, 0].set_ylabel('Mean Discharge (m\u00b3/s)')
    axes[1, 0].set_title('Seasonal Flow Regime')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Flow duration curve (bottom right)
    obs_sorted = obs_clean.sort_values(ascending=False)
    sim_sorted = sim_clean.sort_values(ascending=False)
    obs_ranks = np.arange(1., len(obs_sorted) + 1) / len(obs_sorted) * 100
    sim_ranks = np.arange(1., len(sim_sorted) + 1) / len(sim_sorted) * 100
    axes[1, 1].semilogy(obs_ranks, obs_sorted, 'b-', label='Observed', linewidth=2)
    axes[1, 1].semilogy(sim_ranks, sim_sorted, 'r-', label='Simulated', linewidth=2)
    axes[1, 1].set_xlabel('Exceedance Probability (%)')
    axes[1, 1].set_ylabel('Discharge (m\u00b3/s)')
    axes[1, 1].set_title('Flow Duration Curve')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(f'Logan River at Logan \u2014 Lumped {model_name} Evaluation', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping visualization (no observation data available)")

print("\nStreamflow evaluation complete")

## Step 5b — Model Calibration

Calibrate model parameters using the DDS (Dynamically Dimensioned Search) algorithm to improve model performance. The calibration optimizes KGE over the calibration period.

> **Note:** Calibration parameters are model-specific. SUMMA uses its built-in parameter set. JAX models (HBV, SAC-SMA, etc.) use their own parameter managers registered via the plugin system.

In [ ]:
# Step 5b — Run calibration
print(f"Starting calibration...")
print(f"Algorithm: {config.optimization.algorithm}")
print(f"Metric: {config.optimization.metric}")
print(f"Iterations: {config.optimization.iterations}")
print(f"Calibration period: {config.domain.calibration_period}")

results_file = symfluence.managers['optimization'].calibrate_model()
print(f"\nCalibration complete!")
print(f"Results file: {results_file}")

### View Calibration Results

In [ ]:
# Load and display calibration results
if results_file and Path(results_file).exists():
    results_df = pd.read_csv(results_file)
    
    # Compute running best score (cumulative max for KGE)
    results_df['best_score'] = results_df['score'].cummax()
    
    print("Calibration Progress:")
    print(f"  Best {config.optimization.metric}: {results_df['best_score'].iloc[-1]:.4f}")
    print(f"  Initial {config.optimization.metric}: {results_df['best_score'].iloc[0]:.4f}")
    print(f"  Improvement: {results_df['best_score'].iloc[-1] - results_df['best_score'].iloc[0]:.4f}")
    
    # Plot calibration progress
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(results_df['iteration'], results_df['best_score'], 'b-', linewidth=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(f'Best {config.optimization.metric}')
    ax.set_title('Calibration Progress')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No calibration results found.")